# **LLM Pipeline to Process Extracted Dataset (Phase 1)**

In [ ]:
!pip install pandas

### **Start custom worker on MultiCare dataset**

In [ ]:
from DatasetWorker import mount_worker
from DatasetWorkerHelpers import get_secret

DATASET_NAME = "extracted_dataset_1792212375"
worker = mount_worker(DATASET_NAME)

### **Clean up junk files (safe to run everytime)**

In [ ]:
worker.clean_dataset()

### **Eliminate cases with ambiguous demographic**

In [ ]:
worker.delete_unknown_demographics()

### **Generate .txt queries for every cases.csv found**

In [ ]:
worker.generate_diagnosis_questions()

### **Attach scores to cases (requiring prompt 3 and result.json)**

In [ ]:
worker.attach_scores()

### **Show dataset quality metrics**

In [ ]:
worker.display_metrics()

# **Final Processing (Split into train and test)**

### **Build big dataset (Generate `normalized_diseases.csv` and `evaluated_cases.csv`)**

In [ ]:
worker.build_master_dataset()

### **Split into train and test. Mask test set (use prompt 4)**

In [7]:
train_df, test_df = worker.assign_splits()

masked_df = worker.get_masked_test_cases()

### **Upload to HuggingFace**

In [ ]:
from huggingface_hub import login, whoami

# Retrieve token and set env var in one line
hf_token = get_secret("HF_TOKEN")

login(token=hf_token)
print(whoami())

In [5]:
from DatasetWorkerHelpers import build_hf_dataset

dataset_dict = build_hf_dataset(train_df=train_df, test_df=test_df, masked_df=masked_df)
REPO_ID = "choisongjun/Tropical_Diseases_Repo_v3.0"

In [ ]:
dataset_dict.push_to_hub(REPO_ID, token=hf_token)
print(f"\nPushed to https://huggingface.co/datasets/{REPO_ID}")